# Equation Symbol CNN — train on Kaggle GPU

Classifies handwritten digits 0-9 and operators + - * / (14 classes) from
the `sagyamthapa/handwritten-math-symbols` dataset.

Augmentation is in three groups:

1. **Custom real-world damage** — random occlusions, partial erasures,
   noise injection.
2. **Geometric jitter** — rotation, translation, scale, perspective,
   random erasing (ported from the MNIST CNN in `CNN_PyTorch/`).
3. **Photometric jitter** — brightness/contrast/blur. The MNIST version
   had no equivalent because MNIST is clean and digitally rendered; this
   model gets pointed at phone photos of paper, where exposure and focus
   vary a lot.

**Before running:** Add Data -> `sagyamthapa/handwritten-math-symbols`.
Settings -> Accelerator -> GPU. Run All.

Outputs `equation_cnn.pth`, `classes.json`, and `test_split_manifest.txt`
to `/kaggle/working/`.


In [ ]:
import os, json, random
import numpy as np
from PIL import Image, ImageDraw, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
CLASSES = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "add", "sub", "mul", "div"]
SYMBOL_DISPLAY = {"add": "+", "sub": "-", "mul": "*", "div": "/"}
IMG_SIZE = 45
FILL = 255  # white paper

# Measured over all 7,750 images of this dataset after base_transform().
# The MNIST-era code hardcoded MNIST's (0.1307, 0.3081), which is inverted
# and wrong here: this data is mostly white paper, so the mean is high.
NORM_MEAN = 0.9435
NORM_STD = 0.2053


def find_data_root():
    # Kaggle's /kaggle/input mount layout has changed across CLI/platform
    # versions (older: /kaggle/input/<slug>/..., newer:
    # /kaggle/input/datasets/<owner>/<slug>/...), so search rather than
    # hardcode either layout.
    fingerprint = {"add", "sub", "mul", "div", "0", "1"}
    for root, dirs, _files in os.walk("/kaggle/input"):
        if fingerprint.issubset(set(dirs)):
            return root
    raise FileNotFoundError(
        "could not locate the handwritten-math-symbols dataset under /kaggle/input -- did you Add Data?"
    )


DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)


In [ ]:
def to_square(img, fill=255):
    w, h = img.size
    side = max(w, h)
    return ImageOps.pad(img, (side, side), color=fill, centering=(0.5, 0.5))


def stretch_contrast(pil_img, cutoff=2):
    """Rescale so the symbol's own dark/light range spans the full 0-255.

    Applied at BOTH train and inference so it acts as a domain-invariance
    step, not a test-time hack: dataset scans already span the full range
    (near no-op), while phone photos of paper never reach true black/white
    under ambient light.
    """
    arr = np.asarray(pil_img, dtype=np.float32)
    lo, hi = np.percentile(arr, [cutoff, 100 - cutoff])
    if hi - lo < 1.0:
        return pil_img
    out = np.clip((arr - lo) * (255.0 / (hi - lo)), 0, 255).astype(np.uint8)
    return Image.fromarray(out)


def base_transform(pil_img):
    """Deterministic geometry+contrast normalization applied everywhere."""
    img = pil_img.convert("L")
    img = to_square(img, fill=255)
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    img = stretch_contrast(img)
    return img


In [ ]:
class RandomOcclusion:
    """Small random black/white blobs -- ink smudges, or something
    partially covering the symbol."""

    def __init__(self, p=0.3, max_patches=2, max_frac=0.18):
        self.p, self.max_patches, self.max_frac = p, max_patches, max_frac

    def __call__(self, img):
        if random.random() > self.p:
            return img
        img = img.copy()
        w, h = img.size
        draw = ImageDraw.Draw(img)
        for _ in range(random.randint(1, self.max_patches)):
            pw = random.uniform(0.08, self.max_frac) * w
            ph = random.uniform(0.08, self.max_frac) * h
            x0 = random.uniform(0, w - pw)
            y0 = random.uniform(0, h - ph)
            draw.rectangle([x0, y0, x0 + pw, y0 + ph], fill=random.choice([0, 255]))
        return img


class RandomPartialErasure:
    """Erase a chunk near one edge -- a partially erased or cut-off stroke."""

    def __init__(self, p=0.25, max_frac=0.25):
        self.p, self.max_frac = p, max_frac

    def __call__(self, img):
        if random.random() > self.p:
            return img
        img = img.copy()
        w, h = img.size
        draw = ImageDraw.Draw(img)
        side = random.choice(["left", "right", "top", "bottom"])
        frac = random.uniform(0.12, self.max_frac)
        if side == "left":
            box = [0, 0, w * frac, h]
        elif side == "right":
            box = [w * (1 - frac), 0, w, h]
        elif side == "top":
            box = [0, 0, w, h * frac]
        else:
            box = [0, h * (1 - frac), w, h]
        draw.rectangle(box, fill=255)
        return img


class RandomNoiseInjection:
    """Gaussian + salt-and-pepper noise on a [0,1] float tensor."""

    def __init__(self, p=0.4, gaussian_std=0.06, salt_pepper_frac=0.01):
        self.p, self.gaussian_std, self.salt_pepper_frac = p, gaussian_std, salt_pepper_frac

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor
        out = tensor.numpy() + np.random.normal(0, self.gaussian_std, tensor.shape).astype("float32")
        mask = np.random.random(out.shape) < self.salt_pepper_frac
        salt = np.random.random(out.shape) < 0.5
        out[mask & salt] = 1.0
        out[mask & ~salt] = 0.0
        return torch.from_numpy(out.clip(0.0, 1.0).astype("float32"))


# PIL-level augmentation, applied after base_transform().
# fill=FILL on every geometric op: these are dark strokes on white paper,
# so torchvision's default black fill would paste fake black borders in.
train_augment = transforms.Compose([
    RandomOcclusion(p=0.3),
    RandomPartialErasure(p=0.25),
    transforms.RandomRotation(10, fill=FILL),
    transforms.RandomAffine(0, translate=(0.1, 0.1), fill=FILL),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5, fill=FILL),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.1), ratio=(0.9, 1.1), antialias=True),
    transforms.ColorJitter(brightness=0.35, contrast=0.35),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2)),
])

# Tensor-level. RandomErasing uses value=1.0 (white) so it erases *to
# paper*, matching the MNIST version's intent (there, 0 was background).
to_tensor = transforms.ToTensor()
random_erasing = transforms.RandomErasing(p=0.2, scale=(0.02, 0.12), value=1.0)
noise_inject = RandomNoiseInjection(p=0.4)
normalize = transforms.Normalize((NORM_MEAN,), (NORM_STD,))


In [ ]:
class SymbolDataset(Dataset):
    def __init__(self, samples, train=False):
        self.samples = samples
        self.train = train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = base_transform(Image.open(path))
        if self.train:
            img = train_augment(img)
        tensor = to_tensor(img)
        if self.train:
            tensor = random_erasing(tensor)
            tensor = noise_inject(tensor)
        tensor = normalize(tensor)
        return tensor, label


# gather file list per class (sorted for determinism), stratified 80/10/10
all_samples = {cls: [] for cls in CLASSES}
for idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(DATA_ROOT, cls)
    for fname in sorted(os.listdir(cls_dir)):
        # the dataset ships stray '.directory' files that PIL cannot open
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        all_samples[cls].append((os.path.join(cls_dir, fname), idx))

train_samples, val_samples, test_samples = [], [], []
for cls, items in all_samples.items():
    random.shuffle(items)
    n = len(items)
    n_train = int(n * 0.8)
    n_val = int(n * 0.1)
    train_samples += items[:n_train]
    val_samples += items[n_train:n_train + n_val]
    test_samples += items[n_train + n_val:]

print("train/val/test:", len(train_samples), len(val_samples), len(test_samples))

# save the held-out test file list so the local pipeline can build synthetic
# equations strictly from images the model never saw during training
test_manifest = [os.path.basename(p) + "|" + CLASSES[label] for p, label in test_samples]
with open("/kaggle/working/test_split_manifest.txt", "w") as f:
    f.write("\n".join(test_manifest))

train_ds = SymbolDataset(train_samples, train=True)
val_ds = SymbolDataset(val_samples, train=False)
test_ds = SymbolDataset(test_samples, train=False)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)


In [ ]:
# sanity check: look at what the model actually receives after augmentation
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for ax in axes.ravel():
    x, y = train_ds[random.randrange(len(train_ds))]
    img = (x[0].numpy() * NORM_STD) + NORM_MEAN  # undo normalize for display
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(SYMBOL_DISPLAY.get(CLASSES[y], CLASSES[y]))
    ax.axis("off")
plt.suptitle("augmented training samples")
plt.tight_layout()
plt.show()


In [ ]:
class EquationCNN(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.fc1 = nn.Linear(128 * 5 * 5, 256)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.bn1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.bn2(self.conv2(x))), 2)
        x = F.max_pool2d(F.relu(self.bn3(self.conv3(x))), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = EquationCNN(num_classes=len(CLASSES)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
criterion = nn.CrossEntropyLoss()


In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total


# more epochs than the 20 used before: the augmentation is much heavier now
# (perspective/scale/photometric on top of the originals), so the model needs
# longer to converge and overfits far less.
EPOCHS = 45
best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_acc)
    print(f"epoch {epoch:02d}  train_loss {train_loss:.4f} train_acc {train_acc:.4f}  "
          f"val_loss {val_loss:.4f} val_acc {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/equation_cnn.pth")

print("best val acc:", best_val_acc)


In [ ]:
model.load_state_dict(torch.load("/kaggle/working/equation_cnn.pth", map_location=device))
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"test_loss {test_loss:.4f} test_acc {test_acc:.4f}")

model.eval()
class_correct = {c: 0 for c in CLASSES}
class_total = {c: 0 for c in CLASSES}
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(1)
        for p, t in zip(preds.cpu().tolist(), y.cpu().tolist()):
            cls = CLASSES[t]
            class_total[cls] += 1
            if p == t:
                class_correct[cls] += 1

for c in CLASSES:
    acc = class_correct[c] / max(class_total[c], 1)
    print(f"{c:5s} ({SYMBOL_DISPLAY.get(c, c)}): {acc:.3f}  n={class_total[c]}")


In [ ]:
with open("/kaggle/working/classes.json", "w") as f:
    json.dump(CLASSES, f)

print("saved equation_cnn.pth, classes.json, test_split_manifest.txt to /kaggle/working/")
